# Tutorial 3 — Turn detection candidates into reviewable alerts (25 minutes)

Run all cells once. This notebook stands alone and uses completed reference mappings.
You will join inventory and identity context, suppress known platform workloads, and examine
the false-positive tradeoff of your own tuning rule. Context is evidence, not a verdict.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts at the repository root or inside tutorials/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "workshop" / "db.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start JupyterLab from the workshop checkout: ./setup.sh --notebooks")
sys.path.insert(0, str(ROOT))
from workshop.tutorial import open_workshop, q, compare, evaluate

con = open_workshop(mappings="solutions")


## 1. Recreate the baseline candidate set


In [ ]:
PRIVILEGED_SQL = """
SELECT metadata.uid AS event_id, cloud.provider AS provider, time,
       actor.user.name AS principal, resources[1].namespace AS namespace,
       json_extract_string(api.request.data, '$.spec.containers[0].image') AS image,
       json_extract_string(api.request.data, '$.metadata.ownerReferences[0].name') AS owner_name,
       json_extract_string(api.request.data, '$.metadata.ownerReferences[0].kind') AS owner_kind,
       cloud.account.uid AS account_uid,
       cloud.account.labels['cluster_name'] AS cluster_name,
       src_endpoint.ip AS src_ip, http_request.user_agent AS user_agent
FROM ocsf_api_activity
WHERE api.operation = 'create'
  AND resources[1].type = 'pods'
  AND status_id = 1
  AND json_extract_string(api.request.data, '$.spec.containers[0].securityContext.privileged') = 'true'
"""
q(con, PRIVILEGED_SQL)


In [ ]:
con.execute("CREATE OR REPLACE TEMP VIEW privileged_hits AS " + PRIVILEGED_SQL)


## 2. Inspect enrichment inputs
All fixture data is synthetic. Allowlists match a provider, namespace, owner kind/name, and image
prefix; identity links connect cloud principals to people. Review those keys before joining.


In [ ]:
q(con, "SELECT * FROM privileged_allowlist")


In [ ]:
q(con, "SELECT * FROM identities")


In [ ]:
q(con, "SELECT * FROM clusters")


## 3. Join context without dropping unmatched events
LEFT JOIN retains hits when inventory is incomplete. Null identity/reputation means unknown,
not safe. Image-prefix checks model a workshop allowlist, not cryptographic image verification.
Client OS mismatch is a weak signal and may have benign explanations.


In [ ]:
ENRICHED_SQL = """
SELECT p.*, w.owner_name IS NOT NULL AS platform_allowlisted,
       c.region, c.data_classification,
       i.display_name, i.out_of_office, i.usual_client_os,
       CASE WHEN i.usual_client_os IS NOT NULL AND p.user_agent IS NOT NULL
            THEN NOT contains(p.user_agent, i.usual_client_os) END AS unusual_client,
       r.reputation, r.as_org,
       NOT EXISTS (SELECT 1 FROM approved_registries a
                   WHERE starts_with(p.image, a.registry_prefix)) AS unapproved_registry
FROM privileged_hits p
LEFT JOIN privileged_allowlist w
  ON w.cloud_provider = p.provider AND w.namespace = p.namespace
 AND w.owner_name = p.owner_name AND w.owner_kind = p.owner_kind
 AND starts_with(p.image, w.image_prefix)
LEFT JOIN clusters c
  ON c.cloud_provider = p.provider AND c.account_uid = p.account_uid
 AND c.cluster_name = p.cluster_name
LEFT JOIN identity_links l
  ON l.cloud_provider = p.provider AND l.principal_name = p.principal
LEFT JOIN identities i ON i.person_id = l.person_id
LEFT JOIN ip_reputation r ON r.ip = p.src_ip
"""
q(con, ENRICHED_SQL)


In [ ]:
con.execute("CREATE OR REPLACE TEMP VIEW enriched_hits AS " + ENRICHED_SQL)


In [ ]:
q(con, """
SELECT provider, platform_allowlisted, count(*) AS candidates
FROM enriched_hits GROUP BY ALL ORDER BY 1, 2
""")


## 4. Suppress known platform workloads, retain evidence
This rule leaves non-allowlisted privileged creates for human review. It is not a claim that every
remaining hit is malicious. The allowlist itself could be wrong or imitated.


In [ ]:
REVIEW_SQL = """
SELECT * FROM enriched_hits WHERE NOT platform_allowlisted
"""
q(con, REVIEW_SQL)


In [ ]:
labels = evaluate(con, REVIEW_SQL)
print("Candidates for review:", len(labels))
print("Known attack hits:", int(labels["is_attack"].sum()))
print("Benign review hits:", int((~labels["is_attack"]).sum()))
labels


## 5. Your tuning rule
The starting query is runnable. Duplicate it and test conditions such as `unapproved_registry`,
`out_of_office`, `unusual_client`, or source IP reputation. First inspect actual values with
GROUP BY queries; do not assume a reputation label's spelling.
Preserve `event_id` so you can compare missed attacks and extra benign hits.


In [ ]:
q(con, "SELECT reputation, count(*) AS hits FROM enriched_hits GROUP BY ALL")


In [ ]:
# YOUR QUERY: add evidence-based conditions; never filter on evaluation ground-truth IDs.
MY_REVIEW_SQL = REVIEW_SQL + " AND reputation = 'malicious'"
q(con, MY_REVIEW_SQL)


In [ ]:
compare(con, MY_REVIEW_SQL, REVIEW_SQL)
tuned_labels = evaluate(con, MY_REVIEW_SQL)
before = set(labels.loc[labels["is_attack"], "event_id"])
after = set(tuned_labels.loc[tuned_labels["is_attack"], "event_id"])
print("Known attack hits lost by tuning:", len(before - after))
tuned_labels


### Counterexample: an approved registry does not imply a safe workload
This deliberately over-restrictive variant keeps only unapproved images. On the bundled
dataset it removes **all three known attack hits**: the malicious activity used approved images.
Treat a lower alert count as a tradeoff to measure, not automatic evidence of a better rule.


In [ ]:
UNAPPROVED_ONLY_SQL = REVIEW_SQL + " AND unapproved_registry"
registry_labels = evaluate(con, UNAPPROVED_ONLY_SQL)
retained = set(registry_labels.loc[registry_labels["is_attack"], "event_id"])
print("Attack hits lost by requiring an unapproved registry:", len(before - retained))
registry_labels


## 6. Explore a second detection: service-account use outside its expected network
This fixture models networks with string prefixes. In production use actual CIDR membership
plus workload identity context. These matches are suspicious candidates, not proof of token theft.


In [ ]:
SERVICE_ACCOUNT_SQL = """
SELECT o.metadata.uid AS event_id, o.cloud.provider AS provider,
               o.actor.user.name AS principal, o.api.operation AS operation,
               o.resources[1].type AS resource_type, o.src_endpoint.ip AS src_ip,
               s.privilege, r.reputation
        FROM ocsf_api_activity o
        JOIN service_accounts s
          ON s.cloud_provider = o.cloud.provider AND s.principal_name = o.actor.user.name
        LEFT JOIN ip_reputation r ON r.ip = o.src_endpoint.ip
        WHERE NOT starts_with(o.src_endpoint.ip, s.in_cluster_ip_prefix)
"""
q(con, SERVICE_ACCOUNT_SQL)


In [ ]:
evaluate(con, SERVICE_ACCOUNT_SQL)


**Final deliverable:** save one SQL detection and a short description containing its intent,
required telemetry, expected benign causes, enrichment joins, and test results. Include at least
one benign example you would retain for review and one extra fixture that tests an edge case.
Do not describe recall against the entire attack chain: these rules target only selected steps.
See `WORKSHOP.md` for sharing SQL in DuckDB UI and the optional mapping/agent exercises.


In [ ]:
# Add your next query below: q(con, "SELECT ...")
